# Data Merging — Stage 3 Cleaning 05: Panel Data Normalisation

## Input
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet`
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_monthly_engineered.parquet`
- `Data/Data_Collection/Final/Stage_3_Cleaning/exclusion_manifest.csv` (from notebook 01, applied here for the first time to the panel-level data)
- `Data/Data_Collection/Final/Stage_4_Normalised/decisions_aggregate.csv` (from notebook 04, needed for the cross-pipeline union step)
- Shared helper modules: `lib.normalise.robust_expanding_zscore`, `lib.normalise.shift_max_stat`, and `lib.review`

## Purpose
The stock-level counterpart to notebook 04. Where notebook 04 normalises and screens the market-level *aggregate* tables (cap-weighted moments across ~100 stocks), this notebook runs the same z-scoring and drop-rule machinery on the underlying **stock-by-stock panel data** (Panel A daily, Panel B monthly) — one row per `(permno, date)` rather than one row per date. It closes a specific known gap: the exclusion manifest built in notebook 01 was applied to the four aggregate tables but never to the panel data itself, so the first block here backfills that.

The notebook ends by reconciling the panel's drop decisions with the aggregate's drop decisions into a single **union** drop list, since both pipelines must ultimately agree on the same base factor set.

## Cell 1 — Backfill Manifest Exclusions, Then Z-Score
- Loads `exclusion_manifest.csv` and drops any matching columns from each panel table before doing anything else — this is the "close the known gap" step. If the manifest is missing, the notebook proceeds unfiltered but warns loudly.
- For each of the two panel tables (`panel_stock_daily_engineered`, bucket `STOCK - DAILY`, min 252 dates; `panel_stock_monthly_engineered`, bucket `STOCK - MONTHLY`, min 24 dates), casts `permno` to `int64` and runs `robust_expanding_zscore` over every remaining feature column, excluding only the relevant meta columns (`permno`, `date`, and the appropriate cap/price columns — there's no target column here, since panel data isn't matched 1:1 with a market return target the way the aggregate tables are).
- As in notebook 04, z-scores are computed causally over the full 2004–2024 span; diagnostics are measured only within the fixed Split-B training window (`rv.DIAG_START`–`rv.DIAG_END`), with a separate NBER crisis window tracked for boundary-mass exclusion checks.
- Saves each z-scored table and accumulates a combined diagnostic report to `panel_zscore_diagnostic_report.csv`.

## Cell 2 — Structural, Rule 4, and Rule 5 Pre-Drops (Panel-Specific)
Runs the same three pre-drop categories as notebook 04, but with panel-specific mechanics:

- **`permno` is loaded explicitly and asserted present** before Rule 5 runs, because Rule 5 differences features *within stock* — without grouping by `permno`, a naive diff would silently subtract one stock's value from a different stock's value at every panel boundary, producing meaningless correlations with no error raised. This is called out as a load-bearing assertion, not a convenience.
- **Structural drops are applied first**, before Rule 4/5 get to adjudicate — otherwise Rule 5's correlation-based tiebreak could sacrifice a genuinely good feature to preserve a column that was already structurally doomed, losing both.
- **Rule 4** (trending level with a `_mom`/`_yoy` twin) is expected to find nothing in the panel, since `_mom`/`_yoy` differenced twins are macro constructs that live only in the aggregate Panels C/D — the printed output documents this absence explicitly rather than silently returning empty.
- **Rule 5** (`|ρ| > 0.995` on first differences) is run with `group_col='permno'` so differencing respects stock boundaries.
- A `rv.PENDING_VERIFICATION` list is printed for manual sign-off before a final run, same as notebook 04.
- Audit trails saved to `rule4_level_twin_audit_panel.csv` and `rule5_duplicate_audit_panel.csv`.

## Cell 3 — Apply Rules and Summarize
Combines the z-score diagnostics with the structural/R4/R5 pre-drops via `rv.apply_rules` (no exemption set here, unlike the aggregate notebook's binary-indicator exemptions — panel data has no such pass-through columns). Saves `decisions_panel.csv` and `surviving_features_panel.csv`, then prints the same drop-count-by-rule-and-bucket summary and primary-reason breakdown as notebook 04, explicitly labelled "before the union" since this cell precedes the cross-pipeline reconciliation.

## Cell 4 — Detailed Drop Listing
Identical structure to notebook 04's Cell 4: for each bucket, every dropped feature is printed grouped by which rule fired, with the full diagnostic row for traceability.

## Cell 5 — Three Validation Checks
The same three sanity checks as notebook 04 (sigma-floor rung crosstab, cap-vs-boundary-mass circularity check, NBER crisis-exclusion rescue count), run here against the panel decisions instead of the aggregate decisions. Check 3 is refined slightly: it explicitly computes both `n_all` and `n_exc` observation counts using the *full* rule (threshold **and** minimum-count requirement) on both sides, so the "rescued" figure is guaranteed to match what Rule 6 (boundary mass) actually fired on rather than an approximation.

## Cell 6 — Union: Reconciling Panel and Aggregate Drop Decisions
This is the structurally important part of the notebook. Both the panel pipeline (this notebook) and the aggregate pipeline (notebook 04) independently decide which base factors to drop, but **both pipelines must end up with the same base factor set**, since the aggregate's `_cwmean` moment is definitionally the same measurement as the panel's underlying column — they must rise or fall together.

The union logic has a specific asymmetry, stated explicitly in the code:

- **Panel → aggregate:** if a base factor is dropped in the *panel*, all five of its aggregate moments (`_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread`) are removed too — because if the underlying stock-level measurement is broken, every derived cross-sectional moment built from it is suspect.
- **Aggregate → panel:** the reverse is *not* unrestricted. A base factor is only added to the union drop list from the aggregate side if its **`_cwmean` specifically** failed (checked via `is_mean` — either the feature is in a means-only table, or it ends in `_cwmean` in a full-moments table). If only a *higher moment* (`_cwstd`, `_cwskew`, `_cwkurt`, `_spread`) failed while the `cwmean` survived, the base factor is **not** removed from the panel — only that specific failing moment is dropped from the aggregate. This is because higher moments are aggregate-only constructs with no panel counterpart, so their failure says nothing about the underlying stock-level data quality.
- `base_factor` is recomputed here using each table's own column list (`rv.base_factor_map`) rather than reusing what `apply_rules` computed without that context — this matters specifically because `_spread`-suffixed columns only correctly strip to their base name when the matching `_cwmean` sibling is visible in the same column set; without it, `X_spread` would map to itself and inflate the "rescued" (higher-moment-only failure) list incorrectly.
- Prints a full breakdown: factors dropped in aggregate only, panel only, both, and the "rescued" list (higher moments failed, cwmean survived — kept in both pipelines).
- Writes `union_drop_list.csv` to **both** `Stage_4_Normalised_Panel` and `Stage_4_Normalised` directories, so neither downstream assembly step needs to reach into the other pipeline's folder.

After computing the union, adds two columns to `decisions_panel.csv`:
- `dropped_by_union` — features this pipeline's own rules kept, but which the aggregate side's failures pull into the union drop list.
- `final_action` — the single column downstream assembly code should read, folding together this pipeline's own verdict and the union outcome, rather than requiring reconstruction from two separate files.

Final surviving feature list (post-union) is saved to `surviving_features_panel.csv` (overwriting the earlier pre-union version from Cell 3).

## Cell 7 — Full-Sample Drift (Reported, Not Acted On)
Same philosophy as notebook 04's final cell: measures `shift_max_z` (drift between early and late sample, computed on the z-scores) for every feature in the **final, post-union** surviving set, and explicitly does not use this to drop anything further. The stated reasoning: `shift_max_z` between early and late sample *is* drift between train and test; dropping on it would selectively remove exactly the features whose behaviour changed over time, which amounts to conditioning the model on knowledge that the future distribution was different — a methodological error. This measurement is deliberately taken **after** the feature set is frozen, so the reported limitation accurately describes what the model actually uses.

Reports drift percentile summary by bucket, cumulative counts above five thresholds (1.0–3.0), and the 25 worst-drifting surviving features.

## Output
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/panel_stock_daily_engineered.parquet`, `panel_stock_monthly_engineered.parquet` — z-scored panel tables.
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/panel_zscore_diagnostic_report.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/rule4_level_twin_audit_panel.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/rule5_duplicate_audit_panel.csv`
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/decisions_panel.csv` — includes `action`, `dropped_by_union`, and `final_action` columns.
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/surviving_features_panel.csv` — post-union final feature list.
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/union_drop_list.csv` **and** `Data/Data_Collection/Final/Stage_4_Normalised/union_drop_list.csv` — identical copies in both pipelines' output directories.
- `Data/Data_Collection/Final/Stage_4_Normalised_Panel/drift_limitation_panel.csv` — full-sample drift on the final feature set, reported as a limitation.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.normalise import robust_expanding_zscore, shift_max_stat
import lib.review as rv

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 400)

IN_DIR   = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
STAGE3   = Path('../../../Data/Data_Collection/Final/Stage_3_Cleaning')
AGG_DIR  = Path('../../../Data/Data_Collection/Final/Stage_4_Normalised')
OUT_DIR  = Path('../../../Data/Data_Collection/Final/Stage_4_Normalised_Panel')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TABLES = {
    'panel_stock_daily_engineered.parquet':
        (252, ['permno', 'date', 'dlyret', 'dlycap'], 'daily', 'STOCK - DAILY'),
    'panel_stock_monthly_engineered.parquet':
        ( 24, ['permno', 'date', 'month_end_cap', 'month_end_price'], 'monthly', 'STOCK - MONTHLY'),
}

BUCKET_ORDER = ['STOCK - DAILY', 'STOCK - MONTHLY']

# ── Close the known gap: 01_apply_exclusions was never applied to the panel ──
manifest_path = STAGE3 / 'exclusion_manifest.csv'
manifest_drop = set()
if manifest_path.exists():
    m = pd.read_csv(manifest_path)
    fcol = next((c for c in ('feature', 'factor', 'column', 'name') if c in m.columns), None)
    if fcol is None:
        raise ValueError(f'exclusion_manifest.csv columns: {list(m.columns)} '
                         f'- no feature column recognised')
    manifest_drop = set(m[fcol].astype(str))
    print(f'exclusion_manifest.csv: {len(manifest_drop)} base factors to drop from the panel')
else:
    print('!! exclusion_manifest.csv not found - panel will NOT be pre-filtered')

print('\n' + '=' * 100)
print('STAGE 3 / 05 - PANEL NORMALISATION')
print('=' * 100)
print(f'  z-scores computed over : ALL rows (2004-2024), expanding, causal')
print(f'  diagnostics measured on: {rv.DIAG_START} .. {rv.DIAG_END}  (Split B training window)')
print(f'  crisis excluded from pct_gt5_ex_crisis: {rv.CRISIS_START} .. {rv.CRISIS_END} (NBER)')

reports, bucket_of, freq_of = [], {}, {}

for fname, (min_dates, meta_cols, freq, bucket) in TABLES.items():
    path = IN_DIR / fname
    if not path.exists():
        print(f'\n  !! {fname} not found - skipped')
        continue

    tag = fname.replace('.parquet', '')
    bucket_of[tag], freq_of[tag] = bucket, freq

    print(f'\n{"-" * 100}\n  {bucket}  |  {tag}  (min_dates={min_dates})')
    df = pd.read_parquet(path)
    df['permno'] = df['permno'].astype('int64')

    hit = sorted(set(df.columns) & manifest_drop)
    if hit:
        df = df.drop(columns=hit)
        print(f'    manifest exclusions applied: {len(hit)} columns removed')

    feature_cols = [c for c in df.columns if c not in meta_cols]
    print(f'    rows {len(df):,}   features {len(feature_cols)}')

    df_z, rep = robust_expanding_zscore(
        df=df, feature_cols=feature_cols, date_col='date',
        min_dates=min_dates,
        diag_start=rv.DIAG_START, diag_end=rv.DIAG_END,
        crisis_start=rv.CRISIS_START, crisis_end=rv.CRISIS_END,
        verbose=True,
    )

    rep.insert(0, 'table_source', tag)
    reports.append(rep)

    df_z.to_parquet(OUT_DIR / fname, index=False, engine='pyarrow')
    print(f'    saved -> {fname}')
    del df, df_z

master = pd.concat(reports, ignore_index=True)
master.to_csv(OUT_DIR / 'panel_zscore_diagnostic_report.csv', index=False)
print(f'\n  diagnostic report: {len(master):,} features')
print(master.groupby('table_source').size().to_string())

exclusion_manifest.csv: 82 base factors to drop from the panel

STAGE 3 / 05 - PANEL NORMALISATION
  z-scores computed over : ALL rows (2004-2024), expanding, causal
  diagnostics measured on: 2004-01-01 .. 2015-12-31  (Split B training window)
  crisis excluded from pct_gt5_ex_crisis: 2007-12-01 .. 2009-06-30 (NBER)

----------------------------------------------------------------------------------------------------
  STOCK - DAILY  |  panel_stock_daily_engineered  (min_dates=252)
    manifest exclusions applied: 23 columns removed
    rows 525,957   features 169
    169 features, 5,285 dates, 525,957 rows (99.5 rows/date)
    z-scored over ALL rows; report measured over 300,430 rows [2004-01-01 .. 2015-12-31]
    crisis rows excluded from pct_gt5_ex_crisis: 39,577 [2007-12-01 .. 2009-06-30]
    sigma_f rung 2:    1 feature(s)  (P95/1.96, >50% ties)
    warm-up trimmed at +/-10 robust sigma: 39,274 values
    contributions capped at +/-10: 142,076   sigma_f refreshes: 20
    std_of_z_

In [3]:
extra_drops = {}
r4_audits, r5_audits = [], []

print('=' * 100)
print('A. STRUCTURAL EXCLUSIONS + RULE 4 + RULE 5   (raw values, diagnostic window)')
print('=' * 100)

for fname, (_, meta_cols, _, bucket) in TABLES.items():
    path = IN_DIR / fname
    if not path.exists():
        continue
    tag = fname.replace('.parquet', '')

    cols_needed = master.loc[master['table_source'] == tag, 'feature'].tolist()

    # permno is REQUIRED: rule5 differences within stock. Without it the diff
    # subtracts one stock's value from another's at every permno boundary --
    # silently, with no error, producing meaningless correlations.
    raw = pd.read_parquet(path, columns=['date', 'permno'] + cols_needed)
    raw = raw[(raw['date'] >= rv.DIAG_START) & (raw['date'] <= rv.DIAG_END)]
    assert 'permno' in raw.columns, 'permno missing - rule5 would diff across stocks'

    # Structural drops first, so R4 and R5 never adjudicate a column that is
    # already leaving. Otherwise R5 can sacrifice a good feature to keep a
    # structurally-doomed one, losing both.
    s = rv.structural_drops(cols_needed)
    for f, v in s.items():
        extra_drops[(tag, f)] = v

    remaining = [c for c in cols_needed if c not in s]

    d4, a4 = rv.rule4(raw, remaining, date_col='date')
    for f, v in d4.items():
        extra_drops[(tag, f)] = v
    if len(a4):
        a4.insert(0, 'table_source', tag)
        r4_audits.append(a4)

    d5, a5 = rv.rule5(raw, remaining, group_col='permno')
    for f, v in d5.items():
        extra_drops.setdefault((tag, f), v)     # structural / R4 take precedence
    if len(a5):
        a5.insert(0, 'table_source', tag)
        r5_audits.append(a5)

    print(f'  {bucket:<16} {tag:<40} structural {len(s):>3}   R4 {len(d4):>3}   R5 {len(d5):>3}')
    del raw

print('\n  ** VERIFY BEFORE FINAL RUN:', rv.PENDING_VERIFICATION, '**')

print('\n' + '=' * 100)
print('RULE 4  - monotone trending level with a _mom / _yoy twin')
print('=' * 100)
r4 = pd.concat(r4_audits, ignore_index=True) if r4_audits else pd.DataFrame()
print(r4.to_string(index=False) if len(r4) else
      '  no level/twin pairs in the panel  (_mom / _yoy twins are macro '
      'constructs and live in Panels C/D)')

print('\n' + '=' * 100)
print('RULE 5  - |rho| > 0.995 on first differences, within stock')
print('=' * 100)
r5 = pd.concat(r5_audits, ignore_index=True) if r5_audits else pd.DataFrame()
print(r5.to_string(index=False) if len(r5) else '  no pairs above 0.995')

r4.to_csv(OUT_DIR / 'rule4_level_twin_audit_panel.csv', index=False)
r5.to_csv(OUT_DIR / 'rule5_duplicate_audit_panel.csv', index=False)

A. STRUCTURAL EXCLUSIONS + RULE 4 + RULE 5   (raw values, diagnostic window)
  STOCK - DAILY    panel_stock_daily_engineered             structural   6   R4   0   R5   6
  STOCK - MONTHLY  panel_stock_monthly_engineered           structural   7   R4   0   R5   3

  ** VERIFY BEFORE FINAL RUN: [] **

RULE 4  - monotone trending level with a _mom / _yoy twin
  no level/twin pairs in the panel  (_mom / _yoy twins are macro constructs and live in Panels C/D)

RULE 5  - |rho| > 0.995 on first differences, within stock
                  table_source                     feature_a                       feature_b  rho_diff                          kept                         dropped  status
  panel_stock_daily_engineered                      turnover                     dvol_to_cap  1.000000                   dvol_to_cap                        turnover dropped
  panel_stock_daily_engineered           buynumtrades_lr_pct            sellnumtrades_lr_pct -1.000000           buynumtrades_lr_pct   

In [4]:
decisions = rv.apply_rules(master, extra_drops, bucket_of, freq_of, exempt=set())
decisions.to_csv(OUT_DIR / 'decisions_panel.csv', index=False)

surv = decisions.loc[decisions['action'] == 'keep', ['table_source', 'feature']]
surv.to_csv(OUT_DIR / 'surviving_features_panel.csv', index=False)

print('=' * 100)
print('DROP COUNTS BY RULE AND BUCKET  (before the union)')
print('=' * 100)
print('A feature can fail several rules, so rule columns do not sum to n_dropped.\n')
print(rv.rule_summary(decisions, BUCKET_ORDER).to_string(index=False))

print('\n' + '-' * 100)
print('PRIMARY REASON (first rule in precedence order)')
print('-' * 100)
prim = (decisions[decisions['action'] == 'drop']
        .groupby(['bucket', 'rule_fired']).size()
        .unstack(fill_value=0).reindex(BUCKET_ORDER))
prim.columns = [rv.RULE_LABEL.get(c, c) for c in prim.columns]
print(prim.T.to_string())

DROP COUNTS BY RULE AND BUCKET  (before the union)
A feature can fail several rules, so rule columns do not sum to n_dropped.

         bucket  n_features  n_dropped  n_kept  S_structural  R4_trending_level_twin  R5_duplicate_rho  R0_unassessable  R1_modal_share  R2_std_bounds  R3_warmup_degenerate  R6_boundary_mass
  STOCK - DAILY         169         19     150             6                       0                 6                0               0              2                     0                 6
STOCK - MONTHLY         185         11     174             7                       0                 3                0               0              0                     0                 3
          TOTAL         354         30     324            13                       0                 9                0               0              2                     0                 9

----------------------------------------------------------------------------------------------------
PRIMARY

In [5]:
SHOW = ['table_source', 'feature', 'rule_fired', 'all_rules',
        'modal_share', 'std_of_z_ex_capped', 'pct_gt5', 'pct_gt5_ex_crisis',
        'n_gt5_ex_crisis', 'sigma_f_rung', 'warmup_degenerate', 'n_obs_diag',
        'max_abs_z', 'shift_max_z', 'd_start', 'reason']

FMT = {'modal_share': '{:.3f}', 'std_of_z_ex_capped': '{:.3f}',
       'pct_gt5': '{:.3%}', 'pct_gt5_ex_crisis': '{:.3%}',
       'max_abs_z': '{:,.1f}', 'shift_max_z': '{:.2f}'}

for b in BUCKET_ORDER:
    d = decisions[(decisions['bucket'] == b) & (decisions['action'] == 'drop')]
    print('\n' + '=' * 130)
    print(f'DROPPED  |  {b}  |  {len(d)} features')
    print('=' * 130)
    if not len(d):
        print('  none')
        continue
    for rule in rv.RULE_ORDER:
        dr = d[d['rule_fired'] == rule]
        if not len(dr):
            continue
        print(f'\n  --- {rv.RULE_LABEL[rule]}   ({len(dr)}) ---')
        print(dr[SHOW].sort_values('feature').to_string(index=False,
              formatters={k: v.format for k, v in FMT.items()}))


DROPPED  |  STOCK - DAILY  |  19 features

  --- Structural (construction)   (6) ---
                table_source               feature   rule_fired                       all_rules modal_share std_of_z_ex_capped pct_gt5 pct_gt5_ex_crisis  n_gt5_ex_crisis  sigma_f_rung  warmup_degenerate  n_obs_diag max_abs_z shift_max_z    d_start                                                                                                         reason
panel_stock_daily_engineered          close_vs_mid S_structural                    S_structural       0.048              0.961  0.941%            0.491%             1142             1              False      272011     339.7        0.07 2004-12-31                                                            [zero_denominator] Same construction as open_vs_mid
panel_stock_daily_engineered           nopt_Parity S_structural S_structural | R6_boundary_mass       0.072              1.515  2.933%            2.864%             6411             1             

In [6]:
print('=' * 100)
print('CHECK 1 - sigma_f rung crosstab on the LOW side of rule 2')
print('=' * 100)
print('Rung 1 = ordinary MAD, so variance genuinely fell.')
print('Rungs 2/3/4 = floor-dominated, a different failure worth naming separately.\n')
low = decisions[decisions['std_of_z_ex_capped'] < rv.STD_LO]
if len(low):
    print(pd.crosstab(low['bucket'], low['sigma_f_rung'], margins=True).to_string())
else:
    print('  no features below the lower std bound')

print('\n' + '=' * 100)
print('CHECK 2 - does the cap explain the boundary mass?')
print('=' * 100)
print('The cap acts at |z|>10; pct_gt5 counts |z|>5, so the 5-10 band was never')
print('capped. Capping still biases the running sigma low, inflating later z. If')
print('n_capped is near zero for these features, the circularity is not driving')
print('the rule.\n')
r6 = decisions[decisions['all_rules'].str.contains('R6_boundary_mass', regex=False)]
if len(r6):
    print(r6[['bucket', 'feature', 'pct_gt5', 'pct_gt5_ex_crisis',
              'n_gt5_ex_crisis', 'n_capped', 'n_obs_diag', 'max_abs_z',
              'std_of_z_ex_capped']]
          .sort_values('pct_gt5_ex_crisis', ascending=False).head(40).to_string(index=False))
    print(f'\n  n_capped == 0 for {int((r6["n_capped"] == 0).sum())} of {len(r6)} '
          f'features failing R6')
else:
    print('  no features failed R6')

print('\n' + '=' * 100)
print('CHECK 3 - how many features the crisis exclusion rescued')
print('=' * 100)
# Both sides use the FULL rule, threshold AND minimum count, so the two figures
# are comparable and the rescued count matches what R6 actually fired on.
n_all = (decisions['pct_gt5'] * decisions['n_obs_diag']).round()
n_exc = (decisions['pct_gt5_ex_crisis'] * decisions['n_obs_diag_ex_crisis']).round() \
        if 'n_obs_diag_ex_crisis' in decisions.columns else decisions['n_gt5_ex_crisis']

would = (decisions['pct_gt5'] > rv.PCT_GT5_MAX) & (n_all >= rv.PCT_GT5_MIN_COUNT)
does  = (decisions['pct_gt5_ex_crisis'] > rv.PCT_GT5_MAX) & \
        (decisions['n_gt5_ex_crisis'] >= rv.PCT_GT5_MIN_COUNT)

print(f'  fail on all-window pct_gt5   : {int(would.sum())}')
print(f'  fail ex-crisis (the rule)    : {int(does.sum())}')
print(f'  RESCUED by the NBER exclusion: {int((would & ~does).sum())}   <- write-up number')
resc = decisions[would & ~does]
if len(resc):
    print()
    print(resc[['bucket', 'feature', 'pct_gt5', 'pct_gt5_ex_crisis',
                'n_gt5_ex_crisis', 'max_abs_z']]
          .sort_values('pct_gt5', ascending=False).to_string(index=False))

CHECK 1 - sigma_f rung crosstab on the LOW side of rule 2
Rung 1 = ordinary MAD, so variance genuinely fell.
Rungs 2/3/4 = floor-dominated, a different failure worth naming separately.

sigma_f_rung   1  2  All
bucket                  
STOCK - DAILY  1  1    2
All            1  1    2

CHECK 2 - does the cap explain the boundary mass?
The cap acts at |z|>10; pct_gt5 counts |z|>5, so the 5-10 band was never
capped. Capping still biases the running sigma low, inflating later z. If
n_capped is near zero for these features, the circularity is not driving
the rule.

         bucket                       feature  pct_gt5  pct_gt5_ex_crisis  n_gt5_ex_crisis  n_capped  n_obs_diag    max_abs_z  std_of_z_ex_capped
  STOCK - DAILY               csize_to_shrout 0.046918           0.040270             9366      5198      272153  3084.019595            1.540847
STOCK - MONTHLY                         VarCF 0.031814           0.033225              326       262       11693 31778.124801            0.9

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# UNION
# ═══════════════════════════════════════════════════════════════════════════════
# Both pipelines must end with the same BASE factor set. The union is a column
# selection on the saved parquets, not a re-normalisation: normalise.py keeps a
# separate warm-up, sigma_f and Welford state per feature, so dropping one
# column cannot change any other column's z-scores.
#
# Union membership requires the CAP-WEIGHTED MEAN to have failed. The aggregate
# means table and the full-moments _cwmean column are the same measurement, and
# that measurement is what the panel column corresponds to. Higher moments
# (_cwstd, _cwskew, _cwkurt, _spread) are aggregate-only and have no panel
# counterpart, so a cwkurt failure must not delete the base factor from the
# panel -- it deletes only that moment from the aggregate.
#
# The reverse direction is unrestricted: a base factor dropped in the panel
# removes all five of its aggregate moments, because the underlying stock-level
# measurement being broken makes every derived moment suspect.

agg_dec = AGG_DIR / 'decisions_aggregate.csv'
union_drop = set()

if not agg_dec.exists():
    print('!! decisions_aggregate.csv not found - run 04_normalise.ipynb first.')
    print('   Drift below is measured on the pre-union feature set.')
else:
    a = pd.read_csv(agg_dec)

    # Recompute base_factor WITH each table's column list. apply_rules called
    # base_factor() without it, so _spread columns kept their suffix and
    # inflate the "rescued" list below. a_drop is unaffected either way.
    bmaps = {t: rv.base_factor_map(g['feature'])
             for t, g in a.groupby('table_source')}
    a['base_factor'] = [bmaps[t][f] for t, f in zip(a['table_source'], a['feature'])]

    MOMENT_TABLES = {'agg_market_daily_full_moments',
                     'agg_market_monthly_full_moments'}
    in_moments = a['table_source'].isin(MOMENT_TABLES)
    is_mean = (~in_moments) | a['feature'].str.endswith('_cwmean')

    a_drop     = set(a.loc[(a['action'] == 'drop') & is_mean, 'base_factor'])
    a_drop_any = set(a.loc[a['action'] == 'drop', 'base_factor'])
    p_drop     = set(decisions.loc[decisions['action'] == 'drop', 'base_factor'])
    union_drop = a_drop | p_drop

    print('=' * 100)
    print('UNION - base factors')
    print('=' * 100)
    print(f'  dropped in aggregate (cap-weighted mean failed) : {len(a_drop)}')
    print(f'  dropped in panel                               : {len(p_drop)}')
    print(f'  dropped in both                                : {len(a_drop & p_drop)}')
    print(f'  UNION to apply to both pipelines               : {len(union_drop)}')

    rescued = sorted(a_drop_any - a_drop)
    print(f'\n  base factors whose HIGHER MOMENTS failed but whose cwmean survived: '
          f'{len(rescued)}')
    print(f'  Retained in both pipelines; only the failing moments leave the '
          f'aggregate.')
    for f in rescued:
        print(f'    {f}')

    print(f'\n  panel-only  (the aggregate will lose these): {len(p_drop - a_drop)}')
    for f in sorted(p_drop - a_drop):
        print(f'    {f}')

    print(f'\n  aggregate-only (the panel will lose these): {len(a_drop - p_drop)}')
    for f in sorted(a_drop - p_drop):
        print(f'    {f}')

    # Written to BOTH directories so neither assembly step has to reach into
    # the other pipeline's folder.
    u = pd.Series(sorted(union_drop), name='base_factor')
    u.to_csv(OUT_DIR / 'union_drop_list.csv', index=False)
    u.to_csv(AGG_DIR / 'union_drop_list.csv', index=False)
    print(f'\n  saved -> union_drop_list.csv  (both Stage_4 directories)')

# ── final_action: one file that states the outcome ───────────────────────────
# `action` is this pipeline's own verdict; `final_action` folds in the union, so
# assembly reads one column instead of reconstructing from two files.
decisions['dropped_by_union'] = (
    (decisions['action'] == 'keep') & decisions['base_factor'].isin(union_drop))
decisions['final_action'] = np.where(
    (decisions['action'] == 'drop') | decisions['dropped_by_union'], 'drop', 'keep')
decisions.to_csv(OUT_DIR / 'decisions_panel.csv', index=False)

n_union_only = int(decisions['dropped_by_union'].sum())
print(f'\n  panel columns kept by the panel rules but removed by the union: '
      f'{n_union_only}')
print(f'  panel columns surviving everything: '
      f'{int((decisions["final_action"] == "keep").sum())} of {len(decisions)}')

surv_final = decisions.loc[decisions['final_action'] == 'keep',
                           ['table_source', 'feature']]
surv_final.to_csv(OUT_DIR / 'surviving_features_panel.csv', index=False)

# ═══════════════════════════════════════════════════════════════════════════════
# DRIFT - measured AFTER the feature set is frozen, reported, never acted on
# ═══════════════════════════════════════════════════════════════════════════════
# shift_max_z between early and late sample is drift between train and test.
# Dropping on it would discard precisely the features whose behaviour changed,
# which conditions the model on knowing the future was different. Measured on
# the FINAL feature set so the limitation describes what is actually used.

rows = []
for fname, (_, meta_cols, _, bucket) in TABLES.items():
    path = OUT_DIR / fname
    if not path.exists():
        continue
    tag = fname.replace('.parquet', '')
    keep_cols = surv_final.loc[surv_final['table_source'] == tag, 'feature'].tolist()
    if not keep_cols:
        continue

    z = pd.read_parquet(path, columns=['date'] + keep_cols)
    years = pd.DatetimeIndex(z['date']).year.to_numpy()
    for c in keep_cols:
        rows.append({'bucket': bucket, 'table_source': tag, 'feature': c,
                     'shift_max_z_full_sample':
                         shift_max_stat(z[c].to_numpy(dtype=float), years)})
    del z

drift = pd.DataFrame(rows)
drift.to_csv(OUT_DIR / 'drift_limitation_panel.csv', index=False)

print('\n' + '=' * 100)
print('DRIFT ON THE FINAL FEATURE SET - full sample 2004-2024, REPORTED NOT ACTED ON')
print('=' * 100)
print(drift.groupby('bucket')['shift_max_z_full_sample']
      .describe(percentiles=[.5, .75, .9, .95, .99]).to_string())

print()
for thr in (1.0, 1.5, 2.0, 2.5, 3.0):
    n = int((drift['shift_max_z_full_sample'] > thr).sum())
    print(f'  > {thr:<4} {n:>5}  ({n / len(drift):.1%})')

print('\n  worst 25:')
print(drift.nlargest(25, 'shift_max_z_full_sample').to_string(index=False))

UNION - base factors
  dropped in aggregate (cap-weighted mean failed) : 76
  dropped in panel                               : 30
  dropped in both                                : 21
  UNION to apply to both pipelines               : 85

  base factors whose HIGHER MOMENTS failed but whose cwmean survived: 28
  Retained in both pipelines; only the failing moments leave the aggregate.
    AOP
    ChTax
    ConvDebt
    CredRatDG
    DebtIssuance
    DelBreadth
    EquityDuration
    IntanCFP
    MomOffSeason16YrPlus
    PctAcc
    PctTotAcc
    RoE
    ShareRepurchase
    Tax
    buynumtrades_inst50k_pct
    grcapx
    implied_return
    ptg_downside
    ptg_median_implied
    ptg_rev_alignment
    ptg_upside
    realspread_rel_5d
    retail_dv_share
    sellnumtrades_inst50k_pct
    total_dv_lr_to_cap
    total_n_trades_m_pct
    total_trade_inst50k_pct
    total_vol_to_shrout

  panel-only  (the aggregate will lose these): 9
    OptionVolume1
    nbbqty_before_close_to_shrout
    oi_